In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/srujanakedigehalli/indexing/chunks_metadata_indexflatIP.pkl
/kaggle/input/datasets/srujanakedigehalli/indexing/hooks_indexflatIP.index
/kaggle/input/datasets/srujanakedigehalli/top500/top500_popular.csv


In [2]:
# ── Cell 1: Install Dependencies ───────────────────────────────────────────
!pip install groq sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.9 MB/s eta 0:00:00:00:0100:01


In [3]:
# ── Cell 2: Imports & Setup ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import faiss
import pickle
import json
import re
import time
from groq import Groq
from sentence_transformers import SentenceTransformer
from kaggle_secrets import UserSecretsClient

# ── Groq API ────────────────────────────────────────────────────────────────

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("api")

client  = Groq(api_key=secret_value_0)
MODEL   = "llama-3.3-70b-versatile"

def call_llm(prompt, temperature=0.7, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(2)
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < retries - 1:
                print(f"  Rate limit — waiting 15s (attempt {attempt+1}/{retries})...")
                time.sleep(15)
            else:
                raise e

# ── Load RAG Assets ─────────────────────────────────────────────────────────
index_path  = '/kaggle/input/datasets/srujanakedigehalli/indexing/hooks_indexflatIP.index'
chunks_path = '/kaggle/input/datasets/srujanakedigehalli/indexing/chunks_metadata_indexflatIP.pkl'
top500_path = '/kaggle/input/datasets/srujanakedigehalli/top500/top500_popular.csv'

index = faiss.read_index(index_path)
with open(chunks_path, 'rb') as f:
    chunks = pickle.load(f)

df       = pd.read_csv(top500_path)
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

print(f"FAISS index  : {index.ntotal} vectors")
print(f"Chunks       : {len(chunks)}")
print(f"Videos       : {len(df)}")
print(f"Categories   : {sorted(df['category'].dropna().unique().tolist())}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index  : 754 vectors
Chunks       : 754
Videos       : 500
Categories   : ['Autos & Vehicles', 'Comedy', 'Education', 'Entertainment', 'Film & Animation', 'Gaming', 'Howto & Style', 'Music', 'News & Politics', 'Nonprofits & Activism', 'People & Blogs', 'Pets & Animals', 'Science & Technology']


In [4]:
# ── Cell 3: Retrieval Helper ────────────────────────────────────────────────
def retrieve(query, category=None, segment_type='hook', top_k=5):
    """
    Retrieve semantically similar chunks from FAISS.
    Falls back to no category filter if filtered results are empty.
    """
    query_vec          = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k * 6)
    results            = [chunks[i] for i in indices[0] if i < len(chunks)]

    filtered = results
    if segment_type:
        filtered = [r for r in filtered if r.get('segment_type') == segment_type]
    if category:
        cat_filtered = [r for r in filtered if r.get('category') == category]
        filtered     = cat_filtered if cat_filtered else filtered  # fallback

    return filtered[:top_k]

In [5]:
# ── Cell 4: Agent 1 — Trend Scout ──────────────────────────────────────────
def trend_scout(niche, category, top_n=10):
    """
    Analyses top videos in the category to surface engagement patterns
    and title strategies that correlate with viral performance.
    """
    if category in df['category'].values:
        cat_df = df[df['category'] == category]
    else:
        cat_df = df  # fallback to full dataset

    top = cat_df.nlargest(top_n, 'engagement_ratio')

    patterns = {
        "has_question" : int(top['title'].str.contains(r'\?').sum()),
        "has_number"   : int(top['title'].str.contains(r'\d').sum()),
        "has_brackets" : int(top['title'].str.contains(r'[\[\(]').sum()),
        "has_caps"     : int(top['title'].str.isupper().sum()),
    }

    # Ask Groq to interpret what these titles have in common
    titles_str = "\n".join([f"- {t}" for t in top['title'].tolist()])
    prompt = f"""You are a social media trend analyst.

A creator wants to make content in the niche: "{niche}" (Category: {category})

These are the top trending video titles in this category by engagement:
{titles_str}

In 3 bullet points, explain what common strategies these titles use to attract clicks.
Be specific and concise. Do not number the points, use bullet points only."""

    insight = call_llm(prompt, temperature=0.3)

    return {
        "top_titles"     : top['title'].tolist(),
        "avg_engagement" : round(float(top['engagement_ratio'].mean()), 4),
        "avg_views"      : int(top['view_count'].mean()),
        "patterns"       : patterns,
        "trend_insight"  : insight
    }

In [6]:
# ── Cell 5: Agent 2 — Hook Analyser ────────────────────────────────────────
def hook_analyser(niche, topic, category):
    """
    Retrieves real hooks from high-performing videos and identifies
    the persuasion strategies and linguistic patterns they use.
    """
    retrieved = retrieve(topic, category=category, segment_type='hook', top_k=5)

    if not retrieved:
        retrieved = retrieve(niche, segment_type='hook', top_k=5)

    hooks_str = "\n".join([
        f"- [{r.get('engagement_ratio', 0):.3f} engagement] {r.get('text', '')}"
        for r in retrieved
    ])

    prompt = f"""You are analysing high-performing YouTube hooks for the niche: "{niche}"
Topic: "{topic}"

These are real hooks from trending videos with their engagement scores:
{hooks_str}

Identify:
1. The top 3 persuasion strategies used (e.g. curiosity gap, story opening, bold claim)
2. Common linguistic patterns (e.g. second-person address, specific numbers, power words)
3. One sentence summary of what makes these hooks work

Respond in this exact JSON format only, no extra text:
{{
  "strategies": ["strategy 1", "strategy 2", "strategy 3"],
  "linguistic_patterns": ["pattern 1", "pattern 2", "pattern 3"],
  "summary": "one sentence summary"
}}"""

    raw = call_llm(prompt, temperature=0.3)
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    try:
        analysis = json.loads(raw)
    except json.JSONDecodeError:
        analysis = {
            "strategies"         : ["curiosity gap", "direct value", "story opening"],
            "linguistic_patterns": ["second person", "power words", "specificity"],
            "summary"            : "Top hooks use curiosity and direct value promises."
        }

    return {"retrieved_hooks": retrieved, "analysis": analysis}

In [7]:
# ── Cell 6: Agent 3 — Hook Generator ───────────────────────────────────────
def hook_generator(niche, topic, category, analyser_output,
                   trend_data, feedback=None):
    """
    Generates 5 hook variations using RAG context + trend patterns.
    Each hook uses a different strategy for variety.
    """
    hooks_context = "\n".join([
        f"- {r.get('text', '')}" for r in analyser_output['retrieved_hooks']
    ])
    strategies    = analyser_output['analysis']['strategies']
    top_titles    = "\n".join(trend_data['top_titles'][:5])
    feedback_text = f"\nPrevious hooks were rejected. Address this feedback: {feedback}" \
                    if feedback else ""

    prompt = f"""You are an expert content strategist for social media creators.

Niche: "{niche}"
Topic: "{topic}"
Platform: YouTube (also applicable to TikTok and Instagram Reels)

High-performing hooks from real trending videos in this space:
{hooks_context}

Currently trending titles in {category}:
{top_titles}

Proven strategies to apply: {strategies}
{feedback_text}

Generate exactly 5 hook variations for this topic.
Rules:
- Each hook must be speakable in under 15 seconds (30-40 words max)
- Each hook MUST use a different strategy from this list:
  curiosity_gap, bold_stat, story_opening, direct_value, provocative_question
- Write as if speaking directly to the viewer
- Be specific, not generic — avoid vague language
- No hashtags, no emojis

Respond in this exact JSON format only, no extra text:
{{
  "hooks": [
    {{"text": "hook text", "strategy": "strategy_name",
      "why_it_works": "one sentence explanation"}},
    {{"text": "hook text", "strategy": "strategy_name",
      "why_it_works": "one sentence explanation"}},
    {{"text": "hook text", "strategy": "strategy_name",
      "why_it_works": "one sentence explanation"}},
    {{"text": "hook text", "strategy": "strategy_name",
      "why_it_works": "one sentence explanation"}},
    {{"text": "hook text", "strategy": "strategy_name",
      "why_it_works": "one sentence explanation"}}
  ]
}}"""

    raw = call_llm(prompt, temperature=0.8)
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"hooks": [{"text": "Could not generate hooks.",
                           "strategy": "unknown", "why_it_works": "parse error"}]}

In [8]:
# ── Cell 7: Agent 4 — Critic ────────────────────────────────────────────────
def critic(hooks, benchmark_hooks, feedback_history=None):
    """
    Scores each hook on 3 axes and provides actionable feedback.
    Uses rule-based scoring to avoid LLM self-preference bias.
    """
    def rule_score(hook_text):
        hook_text = str(hook_text).strip()
        words     = hook_text.split()
        s         = {}

        # Curiosity gap
        power_words = ['secret','never','always','shocking','truth','mistake',
                       'actually','real','why','how','what','impossible',
                       'finally','worst','best','exposed','warned','reveal']
        s['curiosity_gap'] = min(10, 4
            + (2 if '?' in hook_text else 0)
            + (2 if any(w in hook_text.lower() for w in power_words) else 0)
            + (1 if hook_text.endswith('...') else 0)
            + (1 if hook_text.strip().startswith('I') else 0))

        # Clarity — word count sweet spot
        wc = len(words)
        if   8 <= wc <= 35: s['clarity'] = 9
        elif 5 <= wc <  8:  s['clarity'] = 6
        elif wc > 35:       s['clarity'] = 5
        else:               s['clarity'] = 3

        # Pattern match
        patterns = {
            'has_number'   : any(w.isdigit() for w in words),
            'has_question' : '?' in hook_text,
            'second_person': 'you' in hook_text.lower(),
            'has_contrast' : any(w in hook_text.lower()
                                 for w in ['but','however','except','until','yet']),
            'has_urgency'  : any(w in hook_text.lower()
                                 for w in ['now','today','before','stop','wait','finally']),
            'first_person' : hook_text.lower().startswith('i '),
        }
        s['pattern_match'] = min(10, 4 + sum(1 for v in patterns.values() if v))
        s['avg']           = round((s['curiosity_gap'] + s['clarity'] + s['pattern_match']) / 3, 2)
        return s

    # Generate LLM feedback for hooks below threshold
    hooks_str    = "\n".join([f"{i+1}. [{h['strategy']}] {h['text']}"
                               for i, h in enumerate(hooks)])
    benchmarks   = "\n".join([f"- {r.get('text','')}" for r in benchmark_hooks])
    prior        = f"\nNote — prior feedback not fully addressed: {feedback_history}" \
                   if feedback_history else ""

    feedback_prompt = f"""You are evaluating YouTube hooks for quality.

Hooks to evaluate:
{hooks_str}

High-performing benchmark hooks:
{benchmarks}
{prior}

For each hook, give ONE specific sentence of feedback on how to improve it.
Focus on: specificity, emotional pull, clarity of value proposition.

Respond in this exact JSON format only:
{{
  "feedback": [
    {{"hook_number": 1, "feedback": "specific improvement suggestion"}},
    {{"hook_number": 2, "feedback": "specific improvement suggestion"}},
    {{"hook_number": 3, "feedback": "specific improvement suggestion"}},
    {{"hook_number": 4, "feedback": "specific improvement suggestion"}},
    {{"hook_number": 5, "feedback": "specific improvement suggestion"}}
  ]
}}"""

    raw_feedback = call_llm(feedback_prompt, temperature=0.3)
    raw_feedback = re.sub(r'^```(?:json)?\s*', '', raw_feedback)
    raw_feedback = re.sub(r'\s*```$', '', raw_feedback)

    try:
        feedback_data = json.loads(raw_feedback)
        feedbacks     = {f['hook_number']: f['feedback']
                         for f in feedback_data['feedback']}
    except Exception:
        feedbacks = {i+1: "Improve specificity and emotional pull." for i in range(5)}

    scores = []
    for i, hook in enumerate(hooks):
        s = rule_score(hook['text'])
        scores.append({
            "hook_number"  : i + 1,
            "hook_text"    : hook['text'],
            "strategy"     : hook['strategy'],
            "curiosity_gap": s['curiosity_gap'],
            "clarity"      : s['clarity'],
            "pattern_match": s['pattern_match'],
            "avg"          : s['avg'],
            "feedback"     : feedbacks.get(i + 1, "Looks good.")
        })

    return {"scores": scores}

In [9]:
# ── Cell 8: Agent 5 — Script Writer ────────────────────────────────────────
def script_writer(niche, topic, category, best_hook,
                  trend_data, analyser_output):
    """
    Generates a complete script outline structured around the
    best-performing hook, with body sections and a CTA.
    """
    trend_insight  = trend_data.get('trend_insight', '')
    body_chunks    = retrieve(topic, category=category,
                              segment_type='body', top_k=3)
    body_examples  = "\n".join([f"- {r.get('text','')[:200]}"
                                 for r in body_chunks])

    prompt = f"""You are an expert scriptwriter for social media video content.

Niche: "{niche}"
Topic: "{topic}"
Platform: YouTube (also applicable to TikTok and Instagram Reels)
Best hook chosen: "{best_hook}"

Trend insights for this category:
{trend_insight}

Body content style from high-performing videos in this niche:
{body_examples}

Write a complete video script outline with the following structure:

1. HOOK (0-15 seconds): Use the hook provided above verbatim.
2. CONTEXT (15-45 seconds): Establish why this topic matters to the viewer.
   Make it personal and relatable.
3. MAIN CONTENT (45 seconds - 3 minutes): 3 to 4 key points or story beats.
   Each point should have a clear takeaway.
4. PATTERN INTERRUPT (mid-video): One surprising fact, twist, or re-engagement
   line to retain viewers past the midpoint.
5. CALL TO ACTION (final 20 seconds): One clear ask — subscribe, comment, or
   visit a link. Tie it back to the hook's promise.

Format each section with a clear label and 2-4 sentences of actual script content,
not just descriptions. Write in a conversational, spoken tone.
Do not use bullet points inside the script sections — write as flowing speech."""

    script = call_llm(prompt, temperature=0.75)
    return script

In [10]:
# ── Cell 9: Full Pipeline — Content Brief Generator ────────────────────────
def generate_content_brief(niche, topic, category,
                            threshold=6.5, max_iterations=3):
    """
    Master function. Given a niche and topic, runs all 5 agents
    and returns a complete content brief.

    Args:
        niche     : e.g. "Tech Reviews", "Fitness", "Personal Finance"
        topic     : e.g. "best budget laptops 2025", "how to lose belly fat"
        category  : must match a category in the dataset
        threshold : minimum avg critic score to accept hooks (default 6.5)
        max_iterations: max regeneration loops (default 3)
    """

    print(f"\n{'='*65}")
    print(f"  AI VIRAL ARCHITECT — CONTENT BRIEF GENERATOR")
    print(f"{'='*65}")
    print(f"  Niche    : {niche}")
    print(f"  Topic    : {topic}")
    print(f"  Category : {category}")
    print(f"{'='*65}\n")

    # ── Agent 1: Trend Scout ────────────────────────────────────────────────
    print("[1/5] Trend Scout — analysing category patterns...")
    trends = trend_scout(niche, category)
    print(f"  Avg engagement : {trends['avg_engagement']}")
    print(f"  Title patterns : {trends['patterns']}")
    print(f"  Insight        :\n{trends['trend_insight']}\n")

    # ── Agent 2: Hook Analyser ──────────────────────────────────────────────
    print("[2/5] Hook Analyser — retrieving & analysing real hooks...")
    analyser_out = hook_analyser(niche, topic, category)
    print(f"  Strategies found : {analyser_out['analysis']['strategies']}")
    print(f"  Summary          : {analyser_out['analysis']['summary']}\n")

    # ── Agents 3 + 4: Generate → Critique → Loop ───────────────────────────
    feedback_history = None
    final_hooks      = None
    final_scores     = None

    for iteration in range(1, max_iterations + 1):
        print(f"[3/5] Hook Generator — iteration {iteration}/{max_iterations}...")
        generated    = hook_generator(niche, topic, category,
                                      analyser_out, trends,
                                      feedback=feedback_history)
        hooks        = generated.get('hooks', [])

        print(f"[4/5] Critic — scoring {len(hooks)} hooks...")
        critique     = critic(hooks, analyser_out['retrieved_hooks'],
                              feedback_history)
        scores       = critique['scores']
        avg_scores   = [s['avg'] for s in scores]

        print(f"  Scores this round : {[round(s,1) for s in avg_scores]}")

        final_hooks  = hooks
        final_scores = scores

        if all(s >= threshold for s in avg_scores):
            print(f"  ✓ All hooks passed threshold ({threshold}). "
                  f"Done in {iteration} iteration(s).\n")
            break
        else:
            low = [s for s in scores if s['avg'] < threshold]
            feedback_history = " | ".join([s['feedback'] for s in low])
            if iteration < max_iterations:
                print(f"  ✗ Some hooks below threshold — regenerating...\n")
            else:
                print(f"  ✗ Max iterations reached — using best available.\n")

    # ── Pick best hook for script ───────────────────────────────────────────
    best_score = max(final_scores, key=lambda s: s['avg'])
    best_hook  = best_score['hook_text']
    print(f"  Best hook (score {best_score['avg']}/10):\n  \"{best_hook}\"\n")

    # ── Agent 5: Script Writer ──────────────────────────────────────────────
    print("[5/5] Script Writer — generating full content brief...")
    script = script_writer(niche, topic, category,
                           best_hook, trends, analyser_out)
    print("  ✓ Script outline generated.\n")

    # ── Assemble Final Brief ────────────────────────────────────────────────
    brief = {
        "niche"             : niche,
        "topic"             : topic,
        "category"          : category,
        "trend_insight"     : trends['trend_insight'],
        "hook_strategies"   : analyser_out['analysis']['strategies'],
        "hook_summary"      : analyser_out['analysis']['summary'],
        "hooks"             : final_hooks,
        "hook_scores"       : final_scores,
        "best_hook"         : best_hook,
        "best_hook_score"   : best_score['avg'],
        "script_outline"    : script
    }

    return brief

In [11]:
# ── Cell 10: Print Brief Cleanly ───────────────────────────────────────────
def print_brief(brief):
    """Prints the content brief in a clean, readable format."""

    print("\n" + "="*65)
    print("  CONTENT BRIEF")
    print("="*65)
    print(f"  Niche    : {brief['niche']}")
    print(f"  Topic    : {brief['topic']}")
    print(f"  Category : {brief['category']}")

    print("\n" + "─"*65)
    print("  TREND INSIGHTS")
    print("─"*65)
    print(brief['trend_insight'])

    print("\n" + "─"*65)
    print("  HOOK ANALYSIS")
    print("─"*65)
    print(f"  Strategies : {brief['hook_strategies']}")
    print(f"  Summary    : {brief['hook_summary']}")

    print("\n" + "─"*65)
    print("  HOOK VARIATIONS (5 options)")
    print("─"*65)
    for i, (hook, score) in enumerate(
            zip(brief['hooks'], brief['hook_scores']), 1):
        marker = " ◄ BEST" if hook['text'] == brief['best_hook'] else ""
        print(f"\n  Hook {i} [{hook['strategy']}]{marker}")
        print(f"  \"{hook['text']}\"")
        print(f"  Why it works : {hook.get('why_it_works','')}")
        print(f"  Score        : {score['avg']}/10  "
              f"(Curiosity:{score['curiosity_gap']} "
              f"Clarity:{score['clarity']} "
              f"Pattern:{score['pattern_match']})")
        print(f"  Feedback     : {score['feedback']}")

    print("\n" + "─"*65)
    print("  SCRIPT OUTLINE")
    print("─"*65)
    print(brief['script_outline'])

    print("\n" + "="*65)
    print("  END OF CONTENT BRIEF")
    print("="*65 + "\n")

In [12]:
# ── Cell 11: Save Brief to Files ───────────────────────────────────────────
def save_brief(brief):
    """Saves brief as both JSON and a clean text file."""
    safe_topic = re.sub(r'[^a-z0-9]', '_', brief['topic'].lower())[:40]

    # JSON — machine readable, good for further processing
    json_path = f"/kaggle/working/brief_{safe_topic}.json"
    with open(json_path, 'w') as f:
        json.dump(brief, f, indent=2)

    # Text — human readable, easy to share with the creator
    txt_path  = f"/kaggle/working/brief_{safe_topic}.txt"
    with open(txt_path, 'w') as f:
        f.write(f"CONTENT BRIEF\n")
        f.write(f"{'='*60}\n")
        f.write(f"Niche    : {brief['niche']}\n")
        f.write(f"Topic    : {brief['topic']}\n")
        f.write(f"Category : {brief['category']}\n\n")
        f.write(f"TREND INSIGHTS\n{'─'*60}\n")
        f.write(f"{brief['trend_insight']}\n\n")
        f.write(f"HOOK ANALYSIS\n{'─'*60}\n")
        f.write(f"Strategies : {brief['hook_strategies']}\n")
        f.write(f"Summary    : {brief['hook_summary']}\n\n")
        f.write(f"HOOK VARIATIONS\n{'─'*60}\n")
        for i, (hook, score) in enumerate(
                zip(brief['hooks'], brief['hook_scores']), 1):
            marker = " [BEST]" if hook['text'] == brief['best_hook'] else ""
            f.write(f"\nHook {i} [{hook['strategy']}]{marker}\n")
            f.write(f"\"{hook['text']}\"\n")
            f.write(f"Why it works : {hook.get('why_it_works','')}\n")
            f.write(f"Score        : {score['avg']}/10\n")
            f.write(f"Feedback     : {score['feedback']}\n")
        f.write(f"\nSCRIPT OUTLINE\n{'─'*60}\n")
        f.write(brief['script_outline'])

    print(f"Saved: {json_path}")
    print(f"Saved: {txt_path}")
    return json_path, txt_path

In [13]:
# ── Cell 12: RUN — Change These Inputs ─────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────
#  CHANGE THESE TWO LINES TO TRY DIFFERENT NICHES AND TOPICS
# ──────────────────────────────────────────────────────────────────────────

NICHE    = "Tech Reviews"           # e.g. "Fitness", "Personal Finance",
                                    #       "Gaming", "Cooking", "Travel"

TOPIC    = "best budget laptops 2025"  # specific video topic

CATEGORY = "Science & Technology"   # must match one of:
# 'Autos & Vehicles', 'Comedy', 'Education', 'Entertainment',
# 'Film & Animation', 'Gaming', 'Howto & Style', 'Music',
# 'News & Politics', 'Nonprofits & Activism', 'People & Blogs',
# 'Pets & Animals', 'Science & Technology'

# ── Run the pipeline ────────────────────────────────────────────────────────
brief = generate_content_brief(
    niche     = NICHE,
    topic     = TOPIC,
    category  = CATEGORY,
    threshold = 6.5,
    max_iterations = 3
)

print_brief(brief)
save_brief(brief)


  AI VIRAL ARCHITECT — CONTENT BRIEF GENERATOR
  Niche    : Tech Reviews
  Topic    : best budget laptops 2025
  Category : Science & Technology

[1/5] Trend Scout — analysing category patterns...
  Avg engagement : 0.1604
  Title patterns : {'has_question': 1, 'has_number': 3, 'has_brackets': 2, 'has_caps': 0}
  Insight        :
* Using attention-grabbing and unusual combinations, such as pairing a tech brand (Samsung) with a popular group (BTS) to create curiosity and appeal to a broader audience.
* Incorporating emotive and thought-provoking language, like "The Truth About my Son" or "Is Your Privacy An Illusion?", to create a sense of intrigue and concern that encourages viewers to click.
* Adding eye-catching and sensational elements, like emojis (👔, 🍰, 🤯) or provocative statements ("Glitterbomb Trap Catches Phone Scammer"), to make the titles more engaging and entertaining.

[2/5] Hook Analyser — retrieving & analysing real hooks...
  Strategies found : ['curiosity gap', 'none f

('/kaggle/working/brief_best_budget_laptops_2025.json',
 '/kaggle/working/brief_best_budget_laptops_2025.txt')

In [14]:
# ── Cell 13: Try More Niches ────────────────────────────────────────────────
# Run multiple briefs and save all of them

test_cases = [
    ("Fitness",           "how to lose belly fat at home",       "Howto & Style"),
    ("Personal Finance",  "5 money mistakes to avoid in your 20s","Education"),
    ("Gaming",            "best free games you are not playing",  "Gaming"),
]

all_briefs = []
for niche, topic, category in test_cases:
    brief = generate_content_brief(niche, topic, category)
    print_brief(brief)
    save_brief(brief)
    all_briefs.append(brief)
    time.sleep(3)   # small pause between runs

print(f"\nGenerated {len(all_briefs)} content briefs.")
print("Download all files from the Output tab.")


  AI VIRAL ARCHITECT — CONTENT BRIEF GENERATOR
  Niche    : Fitness
  Topic    : how to lose belly fat at home
  Category : Howto & Style

[1/5] Trend Scout — analysing category patterns...
  Avg engagement : 0.144
  Title patterns : {'has_question': 0, 'has_number': 3, 'has_brackets': 1, 'has_caps': 0}
  Insight        :
* Using attention-grabbing and unexpected combinations, such as a luxury fashion brand (LOUIS VUITTON) paired with a popular K-pop group (BTS), or a unique challenge (Making a $7000 jacket for $7).
* Incorporating popular culture references, like well-known music artists (Dua Lipa, Taylor Swift, Shakira) or trends (#shorts), to create a sense of familiarity and curiosity.
* Employing emotional triggers, such as heartfelt messages (You’re the best mom I could have ever asked for) or transformative stories (FrOM 0% tO 100% reAL QuiCK!!), to pique the viewer's interest and encourage clicks.

[2/5] Hook Analyser — retrieving & analysing real hooks...
  Strategies found :

In [15]:
# ── Cell A: LLM-Based Evaluator ─────────────────────────────────────────────
# Replaces rule-based scoring with Llama 3.3 70B as the judge.
# This avoids the structural ceiling problem of the rule-based scorer
# where hooks using sophisticated strategies were capped at low scores.

def llm_evaluate_hook(hook_text, strategy, topic, category, benchmark_hooks):
    """
    Uses Llama 3.3 70B to evaluate a single hook on 4 axes.
    Benchmark hooks from the RAG corpus provide grounding context
    so the LLM judges relative to real high-performing content,
    not just generic quality.
    """
    benchmarks_str = "\n".join([
        f"- {r.get('text','')} (engagement: {r.get('engagement_ratio',0):.3f})"
        for r in benchmark_hooks[:3]
    ])

    prompt = f"""You are an expert YouTube content strategist evaluating video hooks.

Your task is to score the following hook for a video about: "{topic}" (Category: {category})

Hook to evaluate:
"{hook_text}"
Strategy used: {strategy}

For context, here are real hooks from high-performing videos in this space
(with their actual audience engagement scores):
{benchmarks_str}

Score the hook on EXACTLY these 4 axes from 1 to 10:

1. curiosity_gap (1-10): Does this hook make the viewer NEED to keep watching?
   Does it create an open loop or unanswered question in the viewer's mind?
   10 = irresistible, viewer cannot stop watching. 1 = no reason to continue.

2. clarity (1-10): Is the video topic and value proposition immediately clear?
   Would a viewer know exactly what they will get from this video?
   10 = crystal clear in the first second. 1 = completely confusing.

3. emotional_pull (1-10): Does the hook trigger an emotional response?
   This includes excitement, fear of missing out, relatability, surprise, or humour.
   10 = strong emotional reaction. 1 = completely flat.

4. platform_fit (1-10): Does this hook match the style and pacing of
   high-performing content in the {category} category on YouTube?
   Does it feel native to the platform, not like an advertisement or essay?
   10 = perfectly native. 1 = feels completely out of place.

Also provide:
- one_line_verdict: A single sentence explaining the hook's biggest strength
- improvement: One specific, actionable suggestion to make it better

Respond in this EXACT JSON format only. No extra text, no markdown:
{{
  "curiosity_gap": <integer 1-10>,
  "clarity": <integer 1-10>,
  "emotional_pull": <integer 1-10>,
  "platform_fit": <integer 1-10>,
  "avg": <float rounded to 2 decimal places>,
  "one_line_verdict": "<one sentence>",
  "improvement": "<one specific suggestion>"
}}"""

    raw = call_llm(prompt, temperature=0.1)  # low temp for consistent scoring
    raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    raw = re.sub(r'\s*```$', '', raw)

    try:
        result = json.loads(raw)
        # Recalculate avg ourselves to ensure accuracy
        axes  = ['curiosity_gap', 'clarity', 'emotional_pull', 'platform_fit']
        result['avg'] = round(sum(result[a] for a in axes) / len(axes), 2)
        return result
    except json.JSONDecodeError:
        print(f"  JSON parse error — raw response: {raw[:200]}")
        return {
            "curiosity_gap"   : 5, "clarity"       : 5,
            "emotional_pull"  : 5, "platform_fit"  : 5,
            "avg"             : 5.0,
            "one_line_verdict": "Could not parse LLM response.",
            "improvement"     : "Re-run evaluation."
        }

print("LLM evaluator defined.")

LLM evaluator defined.


In [16]:
# ── Cell B: Run LLM Evaluation on Generated Briefs ─────────────────────────
# Reads the briefs generated by the pipeline and re-scores all hooks
# using the LLM evaluator. Produces a richer evaluation table.

def llm_evaluate_brief(brief):
    """
    Takes a brief dict (output of generate_content_brief) and
    re-scores all 5 hooks using the LLM evaluator.
    Returns an enriched brief with LLM scores added.
    """
    print(f"\nEvaluating: {brief['topic']} [{brief['category']}]")
    print(f"{'─'*55}")

    # Get benchmark hooks from RAG for grounding the evaluation
    benchmark_hooks = retrieve(
        brief['topic'],
        category=brief['category'],
        segment_type='hook',
        top_k=3
    )

    llm_scores = []
    for i, hook in enumerate(brief['hooks'], 1):
        print(f"  Scoring hook {i}/5: [{hook['strategy']}]...")
        score = llm_evaluate_hook(
            hook_text       = hook['text'],
            strategy        = hook['strategy'],
            topic           = brief['topic'],
            category        = brief['category'],
            benchmark_hooks = benchmark_hooks
        )
        llm_scores.append(score)
        print(f"    avg={score['avg']} | curiosity={score['curiosity_gap']} "
              f"clarity={score['clarity']} emotional={score['emotional_pull']} "
              f"platform={score['platform_fit']}")
        time.sleep(3)  # avoid rate limits

    brief['llm_scores'] = llm_scores

    # Find best hook by LLM scoring
    best_idx   = max(range(len(llm_scores)), key=lambda i: llm_scores[i]['avg'])
    brief['llm_best_hook']  = brief['hooks'][best_idx]['text']
    brief['llm_best_score'] = llm_scores[best_idx]['avg']

    return brief


# ── Run on all_briefs from the pipeline ────────────────────────────────────
# Make sure you have run Cell 13 (generate_content_brief test cases) first.
# 'all_briefs' should already exist from that cell.

print("=" * 55)
print("LLM EVALUATION — ALL BRIEFS")
print("=" * 55)

evaluated_briefs = []
for brief in all_briefs:
    evaluated = llm_evaluate_brief(brief)
    evaluated_briefs.append(evaluated)

print("\nAll briefs evaluated.")

LLM EVALUATION — ALL BRIEFS

Evaluating: how to lose belly fat at home [Howto & Style]
───────────────────────────────────────────────────────
  Scoring hook 1/5: [bold_stat]...
    avg=8.0 | curiosity=8 clarity=9 emotional=7 platform=8
  Scoring hook 2/5: [curiosity_gap]...
    avg=7.75 | curiosity=8 clarity=9 emotional=6 platform=8
  Scoring hook 3/5: [story_opening]...
    avg=6.25 | curiosity=6 clarity=4 emotional=8 platform=7
  Scoring hook 4/5: [direct_value]...
    avg=7.75 | curiosity=8 clarity=9 emotional=6 platform=8
  Scoring hook 5/5: [provocative_question]...
    avg=6.75 | curiosity=6 clarity=8 emotional=7 platform=6

Evaluating: 5 money mistakes to avoid in your 20s [Education]
───────────────────────────────────────────────────────
  Scoring hook 1/5: [curiosity_gap]...
    avg=8.0 | curiosity=8 clarity=9 emotional=7 platform=8
  Scoring hook 2/5: [bold_stat]...
    avg=8.0 | curiosity=8 clarity=9 emotional=7 platform=8
  Scoring hook 3/5: [story_opening]...
    avg=8.0

In [17]:
# ── Cell C: Print LLM Evaluation Results ───────────────────────────────────
def print_llm_evaluation(brief):
    """Prints a clean comparison of rule-based vs LLM scores for one brief."""

    print(f"\n{'='*65}")
    print(f"  EVALUATION REPORT: {brief['topic'].upper()}")
    print(f"  Category: {brief['category']}")
    print(f"{'='*65}")

    print(f"\n{'Hook':<6} {'Strategy':<22} {'Rule avg':>9} "
          f"{'LLM avg':>8} {'Curiosity':>10} {'Clarity':>8} "
          f"{'Emotional':>10} {'Platform':>9}")
    print("─" * 85)

    for i, (hook, rs, ls) in enumerate(
        zip(brief['hooks'], brief['hook_scores'], brief['llm_scores']), 1
    ):
        marker = " ◄" if hook['text'] == brief['llm_best_hook'] else "  "
        print(f"  {i}{marker:<5} {hook['strategy']:<22} {rs['avg']:>9.2f} "
              f"{ls['avg']:>8.2f} {ls['curiosity_gap']:>10} "
              f"{ls['clarity']:>8} {ls['emotional_pull']:>10} "
              f"{ls['platform_fit']:>9}")

    print(f"\n  Best hook by LLM score ({brief['llm_best_score']}/10):")
    print(f"  \"{brief['llm_best_hook']}\"")

    print(f"\n  LLM verdicts:")
    for i, (hook, ls) in enumerate(zip(brief['hooks'], brief['llm_scores']), 1):
        print(f"  Hook {i}: {ls['one_line_verdict']}")
        print(f"         Improve: {ls['improvement']}")

for brief in evaluated_briefs:
    print_llm_evaluation(brief)


  EVALUATION REPORT: HOW TO LOSE BELLY FAT AT HOME
  Category: Howto & Style

Hook   Strategy                Rule avg  LLM avg  Curiosity  Clarity  Emotional  Platform
─────────────────────────────────────────────────────────────────────────────────────
  1 ◄    bold_stat                   6.67     8.00          8        9          7         8
  2      curiosity_gap               6.67     7.75          8        9          6         8
  3      story_opening               6.67     6.25          6        4          8         7
  4      direct_value                5.67     7.75          8        9          6         8
  5      provocative_question        6.00     6.75          6        8          7         6

  Best hook by LLM score (8.0/10):
  "I lost 10 pounds in 2 weeks with this workout, it changed my life"

  LLM verdicts:
  Hook 1: The hook's biggest strength is its use of a specific, relatable, and impressive weight loss result that grabs the viewer's attention.
         Improve: 

In [18]:
# ── Cell D: Comparison Table — Rule-based vs LLM ───────────────────────────
# This is the key table for your research paper.
# Shows both scoring methods side by side across all test cases.

rows = []
for brief in evaluated_briefs:
    for i, (hook, rs, ls) in enumerate(
        zip(brief['hooks'], brief['hook_scores'], brief['llm_scores']), 1
    ):
        rows.append({
            "topic"              : brief['topic'],
            "category"           : brief['category'],
            "hook_number"        : i,
            "strategy"           : hook['strategy'],
            "hook_text"          : hook['text'],

            # Rule-based scores
            "rule_curiosity"     : rs['curiosity_gap'],
            "rule_clarity"       : rs['clarity'],
            "rule_pattern"       : rs['pattern_match'],
            "rule_avg"           : rs['avg'],

            # LLM scores
            "llm_curiosity"      : ls['curiosity_gap'],
            "llm_clarity"        : ls['clarity'],
            "llm_emotional_pull" : ls['emotional_pull'],
            "llm_platform_fit"   : ls['platform_fit'],
            "llm_avg"            : ls['avg'],

            "llm_verdict"        : ls['one_line_verdict'],
            "llm_improvement"    : ls['improvement'],
        })

comparison_df = pd.DataFrame(rows)

# ── Summary statistics ──────────────────────────────────────────────────────
print("=" * 60)
print("SUMMARY: Rule-based vs LLM Evaluation")
print("=" * 60)
print(f"\n{'Metric':<25} {'Rule-based':>12} {'LLM':>12}")
print("─" * 50)
print(f"{'Avg overall score':<25} "
      f"{comparison_df['rule_avg'].mean():>12.2f} "
      f"{comparison_df['llm_avg'].mean():>12.2f}")
print(f"{'Avg curiosity gap':<25} "
      f"{comparison_df['rule_curiosity'].mean():>12.2f} "
      f"{comparison_df['llm_curiosity'].mean():>12.2f}")
print(f"{'Avg clarity':<25} "
      f"{comparison_df['rule_clarity'].mean():>12.2f} "
      f"{comparison_df['llm_clarity'].mean():>12.2f}")
print(f"\nLLM-only axes:")
print(f"  Emotional pull avg : {comparison_df['llm_emotional_pull'].mean():.2f}")
print(f"  Platform fit avg   : {comparison_df['llm_platform_fit'].mean():.2f}")

print(f"\nBest strategy by LLM avg score:")
print(comparison_df.groupby('strategy')['llm_avg'].mean().sort_values(ascending=False).to_string())

# ── Save ────────────────────────────────────────────────────────────────────
comparison_df.to_csv('/kaggle/working/llm_evaluation_results.csv', index=False)
print(f"\nSaved to /kaggle/working/llm_evaluation_results.csv")

SUMMARY: Rule-based vs LLM Evaluation

Metric                      Rule-based          LLM
──────────────────────────────────────────────────
Avg overall score                 6.60         7.32
Avg curiosity gap                 5.60         7.47
Avg clarity                       9.00         8.13

LLM-only axes:
  Emotional pull avg : 6.20
  Platform fit avg   : 7.47

Best strategy by LLM avg score:
strategy
bold_stat               7.833333
curiosity_gap           7.833333
story_opening           7.166667
provocative_question    7.000000
direct_value            6.750000

Saved to /kaggle/working/llm_evaluation_results.csv


In [19]:
# ── Cell E: Why LLM Evaluation is Better — Side by Side Analysis ────────────
# Shows concrete examples where LLM and rule-based scores diverge,
# explaining why the difference matters. Good for the paper discussion.

print("=" * 65)
print("DIVERGENCE ANALYSIS: Where LLM and Rule-based Scores Differ Most")
print("=" * 65)

comparison_df['score_diff'] = comparison_df['llm_avg'] - comparison_df['rule_avg']

# Cases where LLM scored significantly higher than rule-based
print("\nCases where LLM scored HIGHER than rule-based (LLM caught quality missed by rules):")
print("─" * 65)
higher = comparison_df[comparison_df['score_diff'] > 0.5].sort_values(
    'score_diff', ascending=False)
for _, row in higher.head(5).iterrows():
    print(f"\n  Hook    : \"{row['hook_text'][:70]}...\"")
    print(f"  Strategy: {row['strategy']}")
    print(f"  Rule avg: {row['rule_avg']:.2f}  →  LLM avg: {row['llm_avg']:.2f}  "
          f"(+{row['score_diff']:.2f})")
    print(f"  Verdict : {row['llm_verdict']}")

# Cases where rule-based scored higher
print("\nCases where Rule-based scored HIGHER (surface patterns without substance):")
print("─" * 65)
lower = comparison_df[comparison_df['score_diff'] < -0.5].sort_values('score_diff')
for _, row in lower.head(3).iterrows():
    print(f"\n  Hook    : \"{row['hook_text'][:70]}...\"")
    print(f"  Strategy: {row['strategy']}")
    print(f"  Rule avg: {row['rule_avg']:.2f}  →  LLM avg: {row['llm_avg']:.2f}  "
          f"({row['score_diff']:.2f})")
    print(f"  Verdict : {row['llm_verdict']}")

print(f"\nConclusion:")
print(f"  Rule-based avg : {comparison_df['rule_avg'].mean():.2f}")
print(f"  LLM avg        : {comparison_df['llm_avg'].mean():.2f}")
print(f"  Mean divergence: {comparison_df['score_diff'].abs().mean():.2f} points")
print(f"\n  The LLM evaluator captures emotional pull and platform fit —")
print(f"  dimensions invisible to surface-feature rule matching.")

DIVERGENCE ANALYSIS: Where LLM and Rule-based Scores Differ Most

Cases where LLM scored HIGHER than rule-based (LLM caught quality missed by rules):
─────────────────────────────────────────────────────────────────

  Hook    : "The average twenty-something owes over four thousand dollars in credit..."
  Strategy: bold_stat
  Rule avg: 5.67  →  LLM avg: 8.00  (+2.33)
  Verdict : The hook's biggest strength is its ability to grab attention with a specific and alarming statistic about credit card debt.

  Hook    : "Get the exact 10-minute workout that helped me lose my belly fat at ho..."
  Strategy: direct_value
  Rule avg: 5.67  →  LLM avg: 7.75  (+2.08)
  Verdict : The hook's biggest strength is its clear and direct promise of a specific solution to the viewer's problem.

  Hook    : "Ninety percent of gamers are missing out on these insane free games..."
  Strategy: bold_stat
  Rule avg: 5.67  →  LLM avg: 7.50  (+1.83)
  Verdict : The hook effectively creates curiosity by highlight